# molscene quickstart

A Python-first, notebook-native molecular scene. molscene parses the structure,
evaluates selections, and generates the 3D geometry itself (in Rust); the browser
renderer (Three.js) just draws it. No native app required.

v0.1 native representations: **spheres** and **sticks**. (cartoon/surface are
coming in a later milestone.)


In [1]:
import molscene as ms

scene = (
    ms.load("1ubq")
    .sticks("protein", color="spectrum")
    .spheres("hetero", color="element")
)
scene

<molscene.Scene: 2 representation(s)>

## Selections

Pass a plain string or build one with the `ms.select` DSL. Selections are parsed
and evaluated natively in Rust: classification macros (`protein`, `water`, ...),
predicates (`chain A`, `resi 10-30`, `element C`, `b > 50`), boolean composition
(`&`, `|`, `~`), spatial operators (`around` / `within` / `expand` / `beyond`),
and aggregation (`byres` / `bychain` / `bymol`). Invalid selections raise
`ValueError`.


In [2]:
# protein residues near the C-terminus (within 6 Å of residue 76),
# expanded to whole residues — composed and evaluated in Rust
near_cterm = (
    ms.select.byres(ms.select.around(ms.select.resi(76), 6)) & ms.select.protein()
)
ms.load("1ubq").spheres(near_cterm, color="chain")

<molscene.Scene: 1 representation(s)>

## Coloring

Coloring is resolved in Rust from the `color=` string. Beyond schemes like `element` / `chain` / `spectrum`, v0.3 adds per-property colormaps (`bfactor` / `occupancy`, e.g. `bfactor:plasma`), `element:cyan` to keep carbons one color, and `scene.set_color(selection, color)` to repaint a sub-selection on top.

In [3]:
# Color every atom by its B-factor through the viridis colormap (auto-ranged
# over the selection), then repaint the C-terminal tail red on top of it.
(
    ms.load("1ubq")
    .sticks(ms.select.protein(), color="bfactor")
    .set_color(ms.select.resi(71, 76), "red")
)

<molscene.Scene: 1 representation(s)>

## Inspect or export

The scene compiles to a renderer-neutral geometry spec (instanced spheres and
cylinders). `export_html` writes a fully self-contained, offline HTML file.


In [4]:
geom = scene.to_geometry()
print("spheres:", len(geom["spheres"]["centers"]))
print("cylinders:", len(geom["cylinders"]["starts"]))

scene.export_html("scene.html")

spheres: 660
cylinders: 1216


'scene.html'